# MEdAI: train the baseline on Colab's free GPU

Run the cells **top to bottom**, one at a time (Shift + Enter).

**Before you start (one time only)**
1. `Runtime → Change runtime type → T4 GPU → Save`.
2. Create a GitHub token so Colab can read the private repo:
   GitHub → Settings → Developer settings → Personal access tokens → **Fine-grained tokens** →
   Generate new token → Repository access: *Only select repositories* → `medai` →
   Permissions: **Contents: Read and write** → Generate. Copy it.
3. In Colab, click the **key icon (Secrets)** on the left → *Add new secret* →
   Name `GITHUB_TOKEN`, Value = the token → switch **Notebook access** on.

**If Colab disconnects:** reconnect, run every cell again from the top.
Training continues from the last finished epoch, because checkpoints are saved to your Google Drive.

## 1 · Check the GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then run again."
print("GPU:", torch.cuda.get_device_name(0))

## 2 · Connect Google Drive (checkpoints are saved here so a disconnect loses nothing)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3 · Get the code

In [ ]:
import os
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
REPO = '/content/medai'
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/praneethnarne/medai {REPO}
else:
    !git -C {REPO} pull -q
%cd {REPO}
!git log --oneline -1

## 4 · Install dependencies (Colab already has PyTorch; this adds the rest)

In [ ]:
!pip install -q -r backend/requirements.txt

## 5 · Download the datasets from the ISIC archive

Takes several minutes. Safe to re-run: images already downloaded are skipped.
The committed `splits.json` files are used as-is, so the split is identical to the one in the repo.

In [ ]:
!python backend/src/data/download.py pad_ufes_20 --workers 16
!python backend/src/data/download.py ham10000 --workers 16
!python backend/src/data/verify.py ham10000 pad_ufes_20

## 6 · Point the run folder at Google Drive

In [ ]:
import pathlib, shutil
RUN = 'ham_effb0_colab'   # change only if you want a separate run
drive_dir = pathlib.Path('/content/drive/MyDrive/medai_runs') / RUN
drive_dir.mkdir(parents=True, exist_ok=True)
local = pathlib.Path('results/runs') / RUN
local.parent.mkdir(parents=True, exist_ok=True)
if local.is_symlink():
    pass
elif local.exists():
    shutil.rmtree(local)
if not local.exists():
    local.symlink_to(drive_dir)
print(local, '->', drive_dir)
print('already in Drive:', sorted(p.name for p in drive_dir.iterdir()))

## 7 · Train

Stage 1 trains only the new output layer (3 epochs); stage 2 unfreezes the whole network (up to 15 epochs,
stops early if validation macro F1 stops improving for 5 epochs).

`--resume` is harmless on the first run (it just starts fresh) and continues from the last epoch after a disconnect.
Keep this browser tab open while it runs.

In [ ]:
!python -u backend/src/train/train.py \
    --collection ham10000 --target lesion_type --arch efficientnet_b0 \
    --epochs-head 3 --epochs-full 15 --workers 2 \
    --run-name {RUN} --resume

## 8 · Evaluate

First on HAM10000's own test split (in-domain), then on real phone photos from PAD-UFES-20 (cross-domain).
The gap between the two is the project's headline result. Report it whatever it says.

In [ ]:
!python backend/src/eval/evaluate.py --run {RUN} --collection ham10000 --split test

In [ ]:
!python backend/src/eval/evaluate.py --run {RUN} --collection pad_ufes_20 --split test --collapse-to-triage

## 9 · Save the results to GitHub

Pushes the run's JSON files (config, history, metrics) and the evaluation results to a branch called `colab-results`.
Safe to run more than once. Run step 6 again afterwards before launching the demo.
The model weights (`best.pt`, `last.pt`) are too large for git and stay in Google Drive under `medai_runs/`.

In [ ]:
out = pathlib.Path('/content/results_push') / RUN
if out.exists():
    shutil.rmtree(out)
out.mkdir(parents=True)
for f in drive_dir.glob('*.json'):
    shutil.copy(f, out / f.name)
if local.is_symlink():              # replace the Drive link with real files for git
    local.unlink()
    shutil.copytree(out, local)     # (already a plain copy if this cell ran before)
!git checkout -q -B colab-results
!git add results/runs/{RUN} results/eval
!git -c user.name="praneethnarne" -c user.email="praneethnarne@users.noreply.github.com" commit -q -m "Add Colab baseline run {RUN}"
!git push -q -f https://{token}@github.com/praneethnarne/medai colab-results
print('Pushed. Open the repo on GitHub and switch to the colab-results branch.')

## 10 · Launch the demo

Installs Gradio and starts the app on the best checkpoint. Look for a line
`Running on public URL: https://....gradio.live` and open it -- that link works
on any device (laptop, phone, projector) for up to 72 hours while this cell runs.
Stop it with the stop button on the cell.

If step 9 was run, the run folder is a plain copy; re-run step 6 first so the
demo reads `best.pt` from Drive.

In [ ]:
!pip install -q gradio
!python backend/src/demo/app.py --run {RUN} --share